# PyWatermark Robust Training on Colab

This notebook is the recommended training path for the current project state. It is structured around a practical progression:

1. mount Drive and sync the repo
2. prepare a small COCO subset in Drive
3. run a smoke test
4. bootstrap a clean model without attacks
5. run robust training from the clean checkpoint
6. continue a stronger robustness-focused run aimed at pushing validation bit accuracy toward the `0.70-0.80` range
7. export training graphs for the report

This does not guarantee `0.70-0.80`, but it is the strongest training path currently supported by the repo.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone or update the repo and install dependencies

Replace `YOUR_REPO_URL` once. After that, rerun this cell whenever you restart the Colab runtime.

In [ ]:
from pathlib import Path

REPO_URL = "YOUR_REPO_URL"
PROJECT_DIR = Path('/content/PyWatermark')

if PROJECT_DIR.exists():
    %cd /content/PyWatermark
    !git pull
else:
    %cd /content
    !git clone {REPO_URL} PyWatermark
    %cd /content/PyWatermark

!pip install -q -r requirements.txt

## 3. Prepare a lightweight COCO subset directly in Drive

This downloads `val2017` from the official COCO zip and creates `train/val/test` under Drive. The default `4000/500/500` split is a good balance for Colab Free.

In [ ]:
%cd /content/PyWatermark
!python -m data.prepare_coco \
    --download-split val2017 \
    --raw-dir /content/drive/MyDrive/PyWatermark/raw \
    --output-root /content/drive/MyDrive/PyWatermark/datasets \
    --train-count 4000 \
    --val-count 500 \
    --test-count 500 \
    --force

## 4. Run a smoke test

Use this once after setup changes. It verifies the dataset, GPU path, logging, and checkpoint writing before you commit to longer runs.

In [ ]:
%cd /content/PyWatermark
!python -m training.train \
    --train-dir /content/drive/MyDrive/PyWatermark/datasets/train \
    --val-dir /content/drive/MyDrive/PyWatermark/datasets/val \
    --test-dir /content/drive/MyDrive/PyWatermark/datasets/test \
    --checkpoint-dir /content/drive/MyDrive/PyWatermark/artifacts/checkpoints_smoke \
    --log-dir /content/drive/MyDrive/PyWatermark/artifacts/logs_smoke \
    --train-batch-size 8 \
    --eval-batch-size 8 \
    --num-workers 2 \
    --run-smoke-test

## 5. Clean bootstrap run

This trains the easier version of the task first: `16` bits, stronger watermark signal, no attacks, and a detection-focused loss. Use this as the foundation for later robust training.

In [ ]:
%cd /content/PyWatermark
!python -m training.train \
    --train-dir /content/drive/MyDrive/PyWatermark/datasets/train \
    --val-dir /content/drive/MyDrive/PyWatermark/datasets/val \
    --test-dir /content/drive/MyDrive/PyWatermark/datasets/test \
    --checkpoint-dir /content/drive/MyDrive/PyWatermark/artifacts/checkpoints_clean \
    --log-dir /content/drive/MyDrive/PyWatermark/artifacts/logs_clean \
    --epochs 10 \
    --train-batch-size 16 \
    --eval-batch-size 16 \
    --num-workers 2 \
    --key-bits 16 \
    --encoder-alpha 0.10 \
    --invisibility-weight 0.25 \
    --detection-weight 8.0 \
    --disable-attacks

## 6. Robust training stage 1

This starts the robust run from the clean `best.pt`. It keeps the same easier `16`-bit setup but turns attacks back on.

In [ ]:
%cd /content/PyWatermark
!python -m training.train \
    --train-dir /content/drive/MyDrive/PyWatermark/datasets/train \
    --val-dir /content/drive/MyDrive/PyWatermark/datasets/val \
    --test-dir /content/drive/MyDrive/PyWatermark/datasets/test \
    --checkpoint-dir /content/drive/MyDrive/PyWatermark/artifacts/checkpoints_robust_16 \
    --log-dir /content/drive/MyDrive/PyWatermark/artifacts/logs_robust_16 \
    --resume /content/drive/MyDrive/PyWatermark/artifacts/checkpoints_clean/best.pt \
    --epochs 20 \
    --train-batch-size 16 \
    --eval-batch-size 16 \
    --num-workers 2 \
    --key-bits 16 \
    --encoder-alpha 0.10 \
    --invisibility-weight 0.25 \
    --detection-weight 8.0

## 7. Robustness-focused continuation run

This is the main cell for pushing bit accuracy higher. It resumes from the current robust checkpoint, increases watermark strength, increases detection pressure, and relaxes the invisibility objective. This is the recommended path when your goal is robustness first and image quality later.

In [ ]:
%cd /content/PyWatermark
!python -m training.train \
    --train-dir /content/drive/MyDrive/PyWatermark/datasets/train \
    --val-dir /content/drive/MyDrive/PyWatermark/datasets/val \
    --test-dir /content/drive/MyDrive/PyWatermark/datasets/test \
    --checkpoint-dir /content/drive/MyDrive/PyWatermark/artifacts/checkpoints_robust_16 \
    --log-dir /content/drive/MyDrive/PyWatermark/artifacts/logs_robust_16 \
    --resume /content/drive/MyDrive/PyWatermark/artifacts/checkpoints_robust_16/latest.pt \
    --epochs 60 \
    --train-batch-size 16 \
    --eval-batch-size 16 \
    --num-workers 2 \
    --key-bits 16 \
    --encoder-alpha 0.15 \
    --invisibility-weight 0.10 \
    --detection-weight 12.0 \
    --disable-amp

## 8. Resume robust training after a disconnect

If the runtime disconnects after stage 7 has already started, rerun cells 1 and 2, then use this resume cell.

In [ ]:
%cd /content/PyWatermark
!python -m training.train \
    --train-dir /content/drive/MyDrive/PyWatermark/datasets/train \
    --val-dir /content/drive/MyDrive/PyWatermark/datasets/val \
    --test-dir /content/drive/MyDrive/PyWatermark/datasets/test \
    --checkpoint-dir /content/drive/MyDrive/PyWatermark/artifacts/checkpoints_robust_16 \
    --log-dir /content/drive/MyDrive/PyWatermark/artifacts/logs_robust_16 \
    --auto-resume \
    --epochs 60 \
    --train-batch-size 16 \
    --eval-batch-size 16 \
    --num-workers 2 \
    --key-bits 16 \
    --encoder-alpha 0.15 \
    --invisibility-weight 0.10 \
    --detection-weight 12.0 \
    --disable-amp

## 9. Export training graphs for the report

This reads the CSV history from the robust run and saves report-ready PNG plots to Drive.

In [ ]:
%cd /content/PyWatermark
!python -m evaluation.plot_training_curves \
    --history /content/drive/MyDrive/PyWatermark/artifacts/logs_robust_16/metrics_history.csv \
    --output-dir /content/drive/MyDrive/PyWatermark/artifacts/plots_robust_16 \
    --title-prefix "PyWatermark Robust 16-bit"

## 10. Optional evaluation cell

Run this after the robust checkpoint reaches a satisfactory validation bit accuracy.

In [ ]:
%cd /content/PyWatermark
!python -m evaluation.evaluate \
    --test-dir /content/drive/MyDrive/PyWatermark/datasets/test \
    --checkpoint /content/drive/MyDrive/PyWatermark/artifacts/checkpoints_robust_16/best.pt \
    --report-dir /content/drive/MyDrive/PyWatermark/artifacts/reports_robust_16 \
    --batch-size 16 \
    --num-workers 2